# Introduction

Introduce the project, and  your approach, talk about the process of how you came up with the metric and some alternatives you may have explored.

# The Metric

Describe your metric, and what features you are measuring. What datasets are you using?

# The Best Neighborhood

## Import Packages and Data Files

In [15]:
# import necessary packages
import numpy as np
import pandas as pd
import plotly.express as px
import json

# import geojson data
with open("neighborhoods.geojson", "r", encoding="utf-8") as f:
    pgh_nbs = json.load(f)

# import neighborhood data
nb = pd.read_csv("neighborhoods.csv")

# crate list of all neighborhood names
names = nb['hood'].unique()
names.sort()

# create initial metric_df from nb data
metric_df = nb[["hood","sqmiles","intptlat10","intptlon10"]]
metric_df.rename(columns={"hood":"neighborhood","intptlat10":"lat","intptlon10":"lon"}, inplace=True)

# import facilities data
fac_df = pd.read_csv("facilities.csv")

# import crime data
crime_df = pd.read_excel("incidents_2024_thru_mar2026.xlsx")

# import tree data


/tmp/ipykernel_15282/649983754.py:20: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



## Process Facilities Dataset

In [16]:
# copy original df
fac = fac_df.copy()

# filter results to only have official neighborhoods (removes 1 row)
fac = fac.loc[fac["neighborhood"].isin(names)]

# filter results to remove rows where "inactive" value isn't "f"
fac = fac.loc[fac["inactive"] == "f"]

# change "type" strings to all have title case
fac["type"] = fac["type"].str.title()

# make list of all different facility types
type_list = list(fac["type"].unique())
# make list of facility values
value_list = [0,1,5,5,0,5,1,0,1,0.1,10,5,0,0,0,5,0,1,0,0,0,0,1]

# combine types and values into dataframe to merge with fac_df
value_df = pd.DataFrame({"type": type_list, "value": value_list})

# merge with fac
fac = pd.merge(fac, value_df, how="left", on="type")

# remove duplicate pool buildings
df_pools = fac.loc[fac.type=="Pool"].drop_duplicates(subset="parcel_id")
df_not_pools = fac.loc[fac.type!="Pool"]
fac = pd.concat([df_pools, df_not_pools])

# total value of facilites per neighborhood
fac = fac.groupby('neighborhood')['value'].sum()
fac = fac.reset_index()

# merge into metric_df
metric_df = pd.merge(metric_df, fac, on="neighborhood", how="outer")

# fill na values with 0
metric_df['value'] = metric_df['value'].fillna(0)

# rename "value" col and create new col adjusted for area
metric_df.rename(columns={"value":"fac_score"}, inplace=True)
metric_df["adj_fac_score"] = metric_df["fac_score"] / metric_df["sqmiles"]

# normalize score
metric_df["adj_fac_score"] = metric_df["adj_fac_score"] / metric_df["adj_fac_score"].max()

## Process Crime Dataset

In [17]:
# copy original df
crime = crime_df.copy()

# get count of crimes by neighborhood
crime_counts = crime.groupby("Neighborhood").size().reset_index(name="crime_count")

# rename col to merge
crime_counts.rename(columns={"Neighborhood":"neighborhood"}, inplace=True)

# merge into metric_df
metric_df = pd.merge(metric_df, crime_counts, on="neighborhood", how="left")

# fill na values with 0
metric_df["crime_count"] = metric_df["crime_count"].fillna(0)

# create new col adjusted for area, and normalized
metric_df["adj_crime_count"] = metric_df["crime_count"] / metric_df["sqmiles"]
metric_df["adj_crime_count"] = metric_df["adj_crime_count"] / metric_df["adj_crime_count"].max()

## Process Trees Dataset

## Combine Datasets

In [18]:
metric_df.describe()

,sqmiles,fac_score,adj_fac_score,crime_count,adj_crime_count
count,90.000000,90.000000,90.000000,90.000000,90.000000
mean,0.615622,5.498889,0.152940,874.066667,0.125605
std,0.481316,7.149927,0.212634,1187.918799,0.149590
min,0.103298,0.000000,0.000000,0.000000,0.000000
25%,0.284284,0.000000,0.000000,271.250000,0.039540
50%,0.443539,2.500000,0.082595,609.000000,0.079975
75%,0.824866,8.775000,0.220744,1053.500000,0.140568
max,2.676605,33.000000,1.000000,9132.000000,1.000000


## Plot Individual Metrics

In [19]:
# create scatter plot showing value per sqmile
fig1 = px.scatter_map(metric_df, color_discrete_sequence=["blue"],
                      size='adj_fac_score',
                      lat="lat",lon="lon",
                      zoom=10, center={"lat":40.4387, "lon":-79.9972},
                      map_style="streets",
                      title = "Neighborhood Facility Value")
fig1.update_layout(margin={"r":0,"l":0,"b":0},
                   title={"xanchor":"center", "x":0.5})
fig1.show()

In [20]:
# create scatter plot showing value per sqmile
fig2 = px.scatter_map(metric_df, color_discrete_sequence=["red"],
                      size='adj_crime_count',
                      lat="lat",lon="lon",
                      zoom=10, center={"lat":40.4387, "lon":-79.9972},
                      map_style="streets",
                      title = "Neighborhood Crime Count")
fig2.update_layout(margin={"r":0,"l":0,"b":0},
                   title={"xanchor":"center", "x":0.5})
fig2.show()

## Plot Combined Metric

## Determine Best Neighborhood

# Conclusion

Reflect on how the data-driving determination of "best neighborhood" is the same or different from your personal favorite neighborhood. Each member of the group should write their own response to this.